# Market Position Based Pricing

Manual-run review script. Two independent pricing layers run in sequence and write side-by-side columns into the review xlsx. **No API push** -- output is a timestamped Excel for manual review.

## Layer 1 - Position x Achievement matrix (Cell 7)

For each (product, cohort), classify the SKU by its position in the market percentile band and yesterday's achievement bucket (`qty_ratio = yest_qty / p80_target`), then move N steps in the SKU's effective tier ladder per the agreed action matrix.

Output columns: `new_price`, `reason`, `steps`.

### Action matrix

| pos | low (<0.9) | mid (0.9-2.0) | high (>=2.0) |
|---|---|---|---|
| min | HOLD | HOLD | +2 |
| 25 | -1 | -1 | +1 |
| 50 | -2 | -1 | +1 |
| 75 | -3 | -3 | HOLD |
| max | -3 | -2 | HOLD |
| Target margin<target | -2 | -2 | +2 |
| Target margin>=target | -2 | -1 | HOLD |
| below_min | exact `commercial_min` | exact `commercial_min` | exact `commercial_min` |

Margin step bound: `0.1 * target_margin <= |delta_margin| <= 0.5 * target_margin`. Hard floor: `0.9 * wac_p` (commercial_min takes precedence).

## Layer 2 - Margin-aware profit optimizer (Cells 4b / 6b / 7b / 7c)

Manual `NMV_TARGET` input (top of Cell 4b). Pulls 30 days of daily sales tagged with the daily `price_position` from `MATERIALIZED_VIEWS.Pricing_data_extraction` and computes per-SKU sales-by-position history (with cat_brand RELATIVE position-index fallback scaled by the SKU's own anchor, capped at `5 x sku_anchor`).

### Core principle

**Lifting price never increases NMV.** For lifts we use the historical NMV at the candidate position (cat_brand fallback scaled by `sku_anchor`) and cap the predicted change at zero, so noisy history can't project a phantom gain. The optimizer then **picks the lift that minimizes the NMV drop** subject to remaining profitable. Drops can grow NMV via real elasticity from history.

### Step range

All moves walk at most `STEP_CAP = 7` tiers up or down in `effective_tiers` per SKU per run.

### Three phases

**Phase A - Sales SKUs** (top-50% cumulative cohort NMV pool): never lifted.
- Find each SKU's **historical sales-max position** (where ITS OWN avg_daily_nmv was highest; no cat_brand fallback for the target).
- If `current_position > sales_max_position` (currently more expensive than the sweet spot) -> drop to the position whose rank is closest to `sales_max_position`, smallest step preferred (within -3 step cap, floor `max(0.9*wac, commercial_min)`).
- Else -> hold (`sales_hold_at_or_below_max`).

**Phase B - Margin lifts** on non-sales SKUs with `qty_ratio > 1` (over-achievers). Lifts use the historical NMV at the new position, capped at zero net change (no phantom gain from a price up).
- For each over-achiever, evaluate +1 to +`STEP_CAP` within `ceiling = market_max`.
- For each candidate, `delta_nmv = min(0, exp_nmv_at_new - exp_nmv_at_cur)`; `delta_profit = exp_nmv_new * new_margin - exp_nmv_cur * cur_margin`.
- Skip candidates with `delta_profit <= 0`.
- Among the remaining, pick the candidate with **highest efficiency = `delta_profit / max(-delta_nmv, eps)`**. When the candidate has zero NMV drop, efficiency is infinite -> always preferred. This explicitly minimizes the NMV drop while keeping the lift profitable.
- `delta_nmv_v2 <= 0` for every Phase B move (it counts against the gap).

**Phase C - Non-sales drops** to close the remaining gap (only if `gap_after_AB > 0`):
- For each non-sales SKU not yet decided, evaluate -1 to -`STEP_CAP` candidates whose new position raises expected NMV (`delta_nmv > 0`) using the historical sales-by-position with scale-aware fallback.
- Sort candidates globally by efficiency = `delta_nmv / max(-delta_profit, eps)` (NMV gained per EGP of profit sacrificed). Greedy until gap closes; one move per SKU.

One move per SKU max across all phases.

### Output columns

`sales_or_margin` (informational), `nmv_30d`, `nmv_share`, `last7_avg_daily_nmv`, `is_top50_cum`, `is_bottom_quartile`, `sku_nmv_anchor`, `new_price_v2`, `delta_v2`, `delta_pct_v2`, `delta_nmv_v2`, `delta_profit_v2`, `reason_v2`.

## Final recommendation (Cell 8)

The export adds a final fallback layer:

- **`final_price`** = `new_price_v2` if v2 acted, else `new_price` (v1 matrix).
- **`final_reason`** = `reason_v2` if v2 acted, else `reason` (v1).
- **`final_source`** = `'v2'` / `'v1'` / `'hold'` - tells you which layer produced the final price for that row.
- **`final_delta`**, **`final_delta_pct`** - delta vs `current_price` for the final price.

So when v2 decides to hold a SKU (e.g. `sales_hold_at_or_below_max`), the v1 matrix decision is used as the actionable price. This means the review xlsx always has a single column (`final_price`) you can use for the eventual push.

### `reason_v2` vocabulary

- `sales_drop_to_<pos>_step_-N_smax=<pos>` - Phase A drop to sales-max
- `sales_hold_at_or_below_max_(cur=<pos>,smax=<pos>)` - Phase A hold (already optimal for sales)
- `sales_hold_no_own_history` - Phase A hold (no usable own history; cat_brand fallback NOT used for sales-max target)
- `sales_below_min_handled_by_matrix` - matrix in Cell 7 lifts to commercial_min; v2 stays out
- `margin_lift_step_+N_to_<pos>_overachiever` - Phase B free-profit lift
- `p3_close_gap_step_-N_to_<pos>` - Phase C drop to close gap
- `v2_held_no_better_move` - non-sales SKU with history but no profitable move
- `none_no_30d_sales` - no 30d sales, optimizer skipped
- `_floored` suffix - chosen price was clamped up to `max(0.9 * wac_p, commercial_min_price)`

## Cell map

| cell | purpose | layer |
|---|---|---|
| 1 | imports + setup | both |
| 2 | source data load (V2 market, WAC, current_prices from cohort_pricing_changes, products, target margins, commercial_min, stocks, margin_tiers) | both |
| 3 | build lookup at (product, cohort) + margin-tier merge | both |
| 4 | yesterday achievement join | layer 1 |
| **4b** | **30d sales-by-position history + NMV_TARGET input** | layer 2 |
| 5 | filter (closing_stock>0 AND opening_stock>0, OR below commercial_min) | both |
| 6 | derive position + ach_bucket | layer 1 |
| **6b** | **elasticity inputs (nmv_share / bottom_quartile / top50_cum) + sales-by-position dictionary with cat_brand fallback** | layer 2 |
| 7 | layer 1 action matrix -> new_price / reason / steps | layer 1 |
| **7b** | **classification labels (sales / margin / none, informational)** | layer 2 |
| **7c** | **3-phase profit-maximizing optimizer -> new_price_v2 / reason_v2 / delta_nmv_v2 / delta_profit_v2** | layer 2 |
| 8 | review export (timestamped xlsx, both layers' columns side by side) | both |

In [ ]:
# =============================================================================
# CELL 1 - IMPORTS + SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake, get_snowflake_timezone
from constants import WAREHOUSE_MAPPING, COHORT_IDS

TIMEZONE = get_snowflake_timezone()
CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}  |  Snowflake TZ: {TIMEZONE}')

In [ ]:
# =============================================================================
# CELL 2 - SOURCE DATA LOAD
# Copies the loader pattern from manual_price_push.ipynb but with two important
# differences:
#   (a) current_price comes from cohort_pricing_changes only (DBDP is stopped).
#   (b) margin tiers are loaded via get_margin_tiers() so the Target case can
#       fall back to wac/(1 - margin_tier_X) prices.
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

# (a) current_price from cohort_pricing_changes (latest row per cohort, product, PU)
print('Loading current prices (cohort_pricing_changes)...')
CURRENT_PRICES_QUERY = f'''
WITH latest_changes AS (
    SELECT
        cpc.cohort_id,
        pup.product_id,
        pup.packing_unit_id,
        pup.basic_unit_count,
        cpc.price AS current_price
    FROM cohort_pricing_changes cpc
    JOIN packing_unit_products pup ON pup.id = cpc.product_packing_unit_id
    WHERE cpc.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
      AND pup.basic_unit_count = 1
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY cpc.cohort_id, pup.product_id, pup.packing_unit_id
        ORDER BY cpc.created_at DESC
    ) = 1
)
SELECT cohort_id, product_id, packing_unit_id, basic_unit_count, current_price
FROM latest_changes
WHERE basic_unit_count = 1
  AND ((product_id = 1309 AND packing_unit_id = 2) OR product_id <> 1309)
'''
df_prices = query_snowflake(CURRENT_PRICES_QUERY)
print(f'  Current prices: {len(df_prices)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins (brand x cat)...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month')
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading commercial min prices...')
COMMERCIAL_MIN_QUERY = f'''
WITH to_remove AS (
    SELECT check_date AS start_date,
           (check_date + INTERVAL '1 month') + 6 AS end_date
    FROM (
        SELECT CASE
                 WHEN DATE_PART('day', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) < 7
                 THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month')
                 ELSE DATE_FROM_PARTS(
                       YEAR(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE),
                       MONTH(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE),
                       1)
               END AS check_date
    )
),
region_mapping AS (
    SELECT * FROM (VALUES
        ('Greater Cairo', 'Cairo'),
        ('Greater Cairo', 'Giza')
    ) x(region, main_region)
),
warehouse_mapping AS (
    SELECT * FROM (VALUES
        ('Cairo', 'Mostorod', 1, 700),
        ('Giza', 'Barageel', 236, 701),
        ('Giza', 'Sakkarah', 962, 701),
        ('Delta West', 'El-Mahala', 337, 703),
        ('Delta West', 'Tanta', 8, 703),
        ('Delta East', 'Mansoura FC', 339, 704),
        ('Delta East', 'Sharqya', 170, 704),
        ('Upper Egypt', 'Assiut FC', 501, 1124),
        ('Upper Egypt', 'Bani sweif', 401, 1126),
        ('Upper Egypt', 'Menya Samalot', 703, 1123),
        ('Upper Egypt', 'Sohag', 632, 1125),
        ('Alexandria', 'Khorshed Alex', 797, 702)
    ) x(region, warehouse_name, warehouse_id, cohort_id)
)
SELECT DISTINCT
    sku_id AS product_id,
    cohort_id,
    min_price AS commercial_min_price
FROM (
    SELECT mp.product_id AS sku_id, mp.region, min_price, wac1, created_at,
           MAX(created_at) OVER (PARTITION BY mp.product_id, mp.region) AS latest_date
    FROM finance.minimum_prices mp
    JOIN finance.all_cogs f ON f.product_id = mp.product_id
        AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
    WHERE is_deleted = 'false'
      AND created_at BETWEEN (SELECT start_date FROM to_remove) AND (SELECT end_date FROM to_remove)
) comm
LEFT JOIN region_mapping rm ON comm.region = rm.region
LEFT JOIN warehouse_mapping wm ON wm.region = COALESCE(rm.main_region, comm.region)
WHERE created_at = latest_date
  AND min_price < wac1 * 1.5
'''
df_commercial_min = query_snowflake(COMMERCIAL_MIN_QUERY)
print(f'  Commercial min prices: {len(df_commercial_min)} rows')

print('Loading stocks (per warehouse, basic unit)...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
      AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

# (b) Margin tiers per (warehouse_id, product_id) - margin VALUES, not prices.
print('Loading margin tiers...')
df_margin_tiers = get_margin_tiers()
print(f'  Margin tiers: {len(df_margin_tiers)} rows')

print('\nAll data loaded.')

In [ ]:
# =============================================================================
# CELL 3 - BUILD LOOKUP (product x cohort) + margin-tier merge
# =============================================================================

# Step 3a - same shape as manual_price_push lookup
df_market_cohorts = expand_to_cohorts(market_data)

lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# brand x cat target margin fallback (and 0.05 default)
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_margin_tgt', 'target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.05)
lookup.drop(columns=[c for c in lookup.columns
                     if c.startswith('target_bm') or c.startswith('cat_target') or c == 'target_margin_tgt'],
            inplace=True, errors='ignore')

# market percentile derivatives from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# current price per (product, cohort) - basic unit
base_prices = (
    df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']]
    .drop_duplicates(subset=['cohort_id', 'product_id'])
)
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# commercial_min_price per (product, cohort)
if 'cohort_id' in df_commercial_min.columns:
    cmin = df_commercial_min[['product_id', 'cohort_id', 'commercial_min_price']].drop_duplicates()
    lookup = lookup.merge(cmin, on=['product_id', 'cohort_id'], how='left')
else:
    lookup['commercial_min_price'] = np.nan

# total stocks per product (display only)
product_stocks = (
    df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    .rename(columns={'stocks': 'total_stocks'})
)
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With current_price: {lookup["current_price"].notna().sum()}')
print(f'  With commercial_min: {lookup["commercial_min_price"].notna().sum()}')

# Step 3b - margin tiers, warehouse -> cohort by MEAN of margin values
margin_tier_cols = [
    'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
    'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2',
]

wh_to_cohort = pd.DataFrame(
    WAREHOUSE_MAPPING,
    columns=['region', 'wh_name', 'warehouse_id', 'cohort_id'],
)[['warehouse_id', 'cohort_id']]

_present_tier_cols = [c for c in margin_tier_cols if c in df_margin_tiers.columns]
df_margin_tiers_cohort = (
    df_margin_tiers.merge(wh_to_cohort, on='warehouse_id', how='inner')
                   .groupby(['cohort_id', 'product_id'], as_index=False)[_present_tier_cols]
                   .mean()
)
lookup = lookup.merge(
    df_margin_tiers_cohort,
    on=['cohort_id', 'product_id'],
    how='left',
)
if 'margin_tier_1' in lookup.columns:
    print(f'  With margin_tier_1: {lookup["margin_tier_1"].notna().sum()}')

In [ ]:
# =============================================================================
# CELL 4 - ACHIEVEMENT JOIN (yesterday)
# Roll up warehouse to cohort by summing yest_qty + p80_target across the
# cohort's warehouses, then recompute qty_ratio at cohort level.
# =============================================================================
ACHIEVMENT_QUERY = open(os.path.join('queries', 'achievment_yasterday.sql'), encoding='utf-8').read()
df_ach = query_snowflake(ACHIEVMENT_QUERY)
print(f'Achievement rows (warehouse, product): {len(df_ach)}')

# achievment_yasterday returns warehouse_id, product_id, ... .
# Roll up to cohort using WAREHOUSE_MAPPING (already loaded above as wh_to_cohort).
#df_ach_w = df_ach.merge(wh_to_cohort, on='warehouse_id', how='inner')

df_ach_cohort = (
    df_ach.groupby(['cohort_id', 'product_id'], as_index=False)
            .agg(yest_qty=('yest_qty', 'sum'),
                 p80_target=('p80_target', 'sum'),
                 opening_stock=('opening_stock', 'sum'),
                 closing_stock=('closing_stock', 'sum'))
)
df_ach_cohort['qty_ratio'] = (
    df_ach_cohort['yest_qty'] /
    df_ach_cohort['p80_target'].replace(0, np.nan)
)

lookup = lookup.merge(df_ach_cohort, on=['cohort_id', 'product_id'], how='left')
lookup['qty_ratio'] = lookup['qty_ratio'].fillna(0)
print(f'Lookup with achievement: {len(lookup)} rows; '
      f'{lookup["closing_stock"].notna().sum()} have closing_stock')

In [ ]:
# =============================================================================
# CELL 4b - 30-DAY SALES-BY-POSITION HISTORY (manual NMV target)
# Pull historical price_position snapshots from MATERIALIZED_VIEWS.Pricing_data_extraction
# and join to daily sales from product_sales_order, then roll up to cohort grain.
# =============================================================================

# >>> EDIT ME: today's national NMV target in EGP <<<
NMV_TARGET = 0.0

# Position-label normalization map (extraction strings -> our labels)
POSITION_LABEL_MAP = {
    'Below Market': 'below_min',
    'Below Min':    'below_min',
    'At Min':       'min',
    'At 25th':      '25',
    'At 50th':      '50',
    'At 75th':      '75',
    'At Max':       'max',
    'Above Market': 'max',
    'No Market Data': 'Target',
}

print(f'NMV target for today: {NMV_TARGET:,.0f} EGP')

print('Loading 30d price_position snapshots from Pricing_data_extraction...')
EXTRACTION_HISTORY_QUERY = f'''
SELECT
    created_at::DATE AS sale_date,
    warehouse_id,
    product_id,
    price_position
FROM MATERIALIZED_VIEWS.Pricing_data_extraction
WHERE created_at::DATE >= DATEADD(day, -30,
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
  AND created_at::DATE <  CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
  AND price_position IS NOT NULL
'''
df_pos_hist = query_snowflake(EXTRACTION_HISTORY_QUERY)
print(f'  Snapshots: {len(df_pos_hist)} rows')

print('Loading 30d daily sales (product_sales_order)...')
DAILY_SALES_QUERY = f'''
WITH parent_whs AS (SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)),
raw AS (
    SELECT pso.product_id,
           pso.warehouse_id,
           CAST(DATE_TRUNC('day', pso.created_at) AS DATE) AS sale_date,
           SUM(pso.purchased_item_count * pso.basic_unit_count) AS qty,
           SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= DATEADD(day, -30,
            CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
      AND so.created_at <  CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
      AND so.sales_order_status_id NOT IN (7, 12)
      AND so.channel IN ('telesales', 'retailer')
      AND pso.purchased_item_count <> 0
    GROUP BY 1, 2, 3
)
SELECT product_id,
       COALESCE(p.parent_id, raw.warehouse_id) AS warehouse_id,
       sale_date,
       SUM(qty) AS qty,
       SUM(nmv) AS nmv
FROM raw
LEFT JOIN parent_whs p ON p.child_id = raw.warehouse_id
GROUP BY 1, 2, 3
'''
df_sales_hist = query_snowflake(DAILY_SALES_QUERY)
print(f'  Daily sales: {len(df_sales_hist)} rows')

# Tag each day's sales with that day's position
df_hist = df_sales_hist.merge(
    df_pos_hist, on=['sale_date', 'warehouse_id', 'product_id'], how='inner'
)
print(f'  Joined sales x position: {len(df_hist)} rows')

# Normalize position label
df_hist['price_position'] = df_hist['price_position'].map(POSITION_LABEL_MAP).fillna(df_hist['price_position'])
unmapped = df_hist[~df_hist['price_position'].isin(['min','25','50','75','max','below_min','Target'])]
if len(unmapped) > 0:
    print(f'  WARNING: {len(unmapped)} rows have unmapped price_position labels:')
    print(unmapped['price_position'].value_counts().head(10))

# Roll up warehouse -> cohort
df_hist = df_hist.merge(wh_to_cohort, on='warehouse_id', how='inner')

# (a) sales-by-position table at sku_cohort grain
sales_by_pos = (
    df_hist.groupby(['cohort_id', 'product_id', 'price_position'], as_index=False)
           .agg(total_nmv=('nmv', 'sum'),
                total_qty=('qty', 'sum'),
                days=('sale_date', 'nunique'))
)
sales_by_pos['avg_daily_nmv'] = sales_by_pos['total_nmv'] / sales_by_pos['days'].replace(0, np.nan)
print(f'  sales_by_pos: {len(sales_by_pos)} rows')

# (b) 30d totals per (cohort_id, product_id)
nmv_30d = (
    df_hist.groupby(['cohort_id', 'product_id'], as_index=False)
           .agg(nmv_30d=('nmv', 'sum'),
                qty_30d=('qty', 'sum'),
                days_with_sales=('sale_date', 'nunique'))
)
print(f'  nmv_30d: {len(nmv_30d)} rows')

# (c) last-7-day daily NMV avg per (cohort_id, product_id)
if len(df_hist) > 0:
    cutoff7 = df_hist['sale_date'].max() - pd.Timedelta(days=6)
    last7 = (
        df_hist[df_hist['sale_date'] >= cutoff7]
            .groupby(['cohort_id', 'product_id'], as_index=False)['nmv'].sum()
    )
    last7['last7_avg_daily_nmv'] = last7['nmv'] / 7.0
    last7 = last7[['cohort_id', 'product_id', 'last7_avg_daily_nmv']]
else:
    last7 = pd.DataFrame(columns=['cohort_id', 'product_id', 'last7_avg_daily_nmv'])
print(f'  last7: {len(last7)} rows')

In [ ]:
# =============================================================================
# CELL 5 - FILTER
# Keep only SKUs that had inventory at both opening and closing of yesterday.
# Achievement query already filters out warehouses with closing or opening = 0,
# so SKUs with NaN closing/opening here were dropped upstream.
# =============================================================================
before = len(lookup)
lookup = lookup[
    (lookup['closing_stock'].fillna(0) > 0) &
    (lookup['opening_stock'].fillna(0) > 0)
].copy()
print(f'Filtered to closing>0 AND opening>0: {before} -> {len(lookup)}')

In [ ]:
# =============================================================================
# CELL 6 - POSITION + ACH BUCKET
# =============================================================================
def derive_position(row):
    cur = row.get('current_price')
    cmin = row.get('commercial_min_price', 0) or 0
    if pd.isna(cur):
        return 'no_price'
    if cmin > 0 and cur < cmin:
        return 'below_min'
    has_market = (
        isinstance(row.get('price_tiers'), list) and len(row['price_tiers']) > 0
        and pd.notna(row.get('market_min'))
    )
    if not has_market:
        return 'Target'
    if cur >= row['market_max']: return 'max'
    if cur >= row['market_75']:  return '75'
    if cur >= row['market_50']:  return '50'
    if cur >= row['market_25']:  return '25'
    return 'min'

def derive_ach_bucket(ach):
    if pd.isna(ach) or ach < 0.9: return 'low'
    if ach < 2.0:                  return 'mid'
    return 'high'

lookup['position']   = lookup.apply(derive_position, axis=1)
lookup['ach_bucket'] = lookup['qty_ratio'].apply(derive_ach_bucket)

print('Position x Ach distribution:')
print(pd.crosstab(lookup['position'], lookup['ach_bucket'], margins=True))

In [ ]:
# =============================================================================
# CELL 6b - ELASTICITY INPUTS + CLASSIFICATION FLAGS
# nmv_share / bottom-quartile / top50-cumulative flags per (cohort, product),
# plus sku_cohort + (cat, brand) sales-by-position dictionaries with fallback.
# =============================================================================

# nmv_share + bottom-quartile flag within each cohort
nmv_30d['cohort_nmv_total'] = nmv_30d.groupby('cohort_id')['nmv_30d'].transform('sum')
nmv_30d['nmv_share'] = nmv_30d['nmv_30d'] / nmv_30d['cohort_nmv_total'].replace(0, np.nan)
nmv_30d['cohort_q1_nmv'] = nmv_30d.groupby('cohort_id')['nmv_30d'].transform(lambda s: s.quantile(0.25))
nmv_30d['is_bottom_quartile'] = nmv_30d['nmv_30d'] <= nmv_30d['cohort_q1_nmv']

# top-50% cumulative NMV flag within each cohort
nmv_30d = nmv_30d.sort_values(['cohort_id', 'nmv_30d'], ascending=[True, False])
nmv_30d['cum_share'] = nmv_30d.groupby('cohort_id')['nmv_share'].cumsum()
nmv_30d['is_top50_cum'] = nmv_30d['cum_share'] <= 0.50

# Merge classification inputs into lookup
lookup = lookup.merge(
    nmv_30d[['cohort_id', 'product_id', 'nmv_30d', 'nmv_share',
             'is_bottom_quartile', 'is_top50_cum']],
    on=['cohort_id', 'product_id'], how='left',
)
lookup = lookup.merge(last7, on=['cohort_id', 'product_id'], how='left')
lookup['nmv_30d'] = lookup['nmv_30d'].fillna(0)
lookup['nmv_share'] = lookup['nmv_share'].fillna(0)
lookup['last7_avg_daily_nmv'] = lookup['last7_avg_daily_nmv'].fillna(0)
lookup['is_bottom_quartile'] = lookup['is_bottom_quartile'].fillna(False)
lookup['is_top50_cum'] = lookup['is_top50_cum'].fillna(False)

# ----- SKU-own absolute NMV per position (sku_cohort grain) -----
sbp_sku = sales_by_pos.set_index(
    ['cohort_id', 'product_id', 'price_position']
)['avg_daily_nmv'].to_dict()

# ----- (cat, brand) RELATIVE position index -----
# This is the key fix for the apples-to-oranges fallback bug:
# raw cat_brand avg_daily_nmv is dominated by big SKUs and is meaningless when
# applied to a small SKU. We instead build a *relative* position index
# (avg_daily_nmv at position P / mean across positions for this cat_brand) and
# multiply by the SKU's own scale anchor when used as fallback.
sbp_cb = (
    sales_by_pos.merge(df_products[['product_id', 'brand', 'cat']],
                       on='product_id', how='left')
                .groupby(['cat', 'brand', 'price_position'], as_index=False)['avg_daily_nmv']
                .mean()
)
sbp_cb['cb_pool_avg'] = sbp_cb.groupby(['cat', 'brand'])['avg_daily_nmv'].transform('mean')
sbp_cb['position_index'] = (
    sbp_cb['avg_daily_nmv'] / sbp_cb['cb_pool_avg'].replace(0, np.nan)
)
cb_position_index = (
    sbp_cb.set_index(['cat', 'brand', 'price_position'])['position_index'].to_dict()
)

# ----- Per-SKU scale anchor (nmv_30d / 30 if available, else last7 avg) -----
lookup['sku_nmv_anchor'] = np.maximum(
    lookup['nmv_30d'].fillna(0) / 30.0,
    lookup['last7_avg_daily_nmv'].fillna(0),
)

# Sanity cap (multiple of own scale) - protects against noisy 1-day fluke estimates
NMV_PREDICTION_CAP = 5.0   # never project more than 5x the SKU's own daily anchor

def expected_daily_nmv_at(cohort, product, cat, brand, position, sku_anchor):
    """Expected daily NMV for this SKU at a hypothetical position bucket.

    Priority:
      1. sku_cohort own history at that position (absolute NMV from this SKU's
         actual sales). Capped at NMV_PREDICTION_CAP * sku_anchor.
      2. cat_brand POSITION INDEX scaled by sku_anchor. The index is relative
         (1.0 = pool average), so the SKU's own scale is preserved.
         Capped at NMV_PREDICTION_CAP * sku_anchor.

    Returns None if no usable signal."""
    cap = NMV_PREDICTION_CAP * float(sku_anchor or 0)
    v = sbp_sku.get((cohort, product, position))
    if v is not None and not pd.isna(v):
        v = float(v)
        return min(v, cap) if cap > 0 else v
    idx = cb_position_index.get((cat, brand, position))
    if idx is not None and not pd.isna(idx):
        scaled = float(sku_anchor or 0) * float(idx)
        return min(scaled, cap) if cap > 0 else scaled
    return None

# Diagnostic: how many SKUs have any usable signal
sku_hist_keys = {(c, p) for (c, p, _) in sbp_sku.keys()}
has_sku_hist = lookup.apply(
    lambda r: (r['cohort_id'], r['product_id']) in sku_hist_keys, axis=1,
)
print(f'Lookup rows with sku_cohort history: {has_sku_hist.sum()} / {len(lookup)}')
print(f'Lookup rows with non-zero sku_nmv_anchor: '
      f'{(lookup["sku_nmv_anchor"] > 0).sum()} / {len(lookup)}')

In [ ]:
# =============================================================================
# CELL 7 - ACTION MATRIX
#   - build_effective_tiers: price_tiers if non-empty, else margin tier prices.
#   - step_in_tiers: move N steps relative to current_price (clamps to ends).
#   - apply_margin_step_bounds: 0.1*tgt <= |delta_margin| <= 0.5*tgt.
#   - compute_action: per-row matrix dispatch.
# Safety nets at the end: 0.9*wac_p hard floor, then commercial_min_price.
# =============================================================================
def build_effective_tiers(row):
    """Sorted ascending list of prices (0.25 EGP rounded).
    Per design: margin MEANs are pre-aggregated at cohort grain in Cell 3, then
    converted to ONE price per tier via wac/(1-mean_margin)."""
    pt = row.get('price_tiers')
    if isinstance(pt, list) and len(pt) > 0:
        return sorted({round(p * 4) / 4 for p in pt if p > 0})
    wac = row.get('wac_p', 0) or 0
    if wac <= 0:
        return []
    margins = []
    for c in ['margin_tier_1','margin_tier_2','margin_tier_3','margin_tier_4',
              'margin_tier_5','margin_tier_above_1','margin_tier_above_2']:
        m = row.get(c)
        if pd.notna(m) and 0 < m < 1:
            margins.append(m)
    return sorted({round(wac / (1 - m) * 4) / 4 for m in margins})


def step_in_tiers(tiers, current, n):
    """Returns (new_price, was_clamped, no_tier_in_direction)."""
    if n == 0 or not tiers:
        return None, False, False
    if n > 0:
        above = [t for t in tiers if t > current]
        if not above:
            return None, False, True
        idx = min(n - 1, len(above) - 1)
        return above[idx], (idx < n - 1), False
    below = [t for t in tiers if t < current][::-1]
    if not below:
        return None, False, True
    idx = min(abs(n) - 1, len(below) - 1)
    return below[idx], (idx < abs(n) - 1), False


def apply_margin_step_bounds(current, candidate, wac, target_margin):
    """Clamp |delta_margin| to [0.1*tgt, 0.5*tgt]. Returns (price, bound_hit)."""
    if candidate is None or wac <= 0 or current <= 0:
        return candidate, None
    cur_m = (current   - wac) / current
    new_m = (candidate - wac) / candidate
    delta = new_m - cur_m
    if delta == 0:
        return candidate, None
    direction = 1 if delta > 0 else -1
    abs_delta = abs(delta)
    min_step  = 0.1 * (target_margin or 0)
    max_step  = 0.5 * (target_margin or 0)
    if min_step <= 0 and max_step <= 0:
        return candidate, None
    if abs_delta < min_step:
        clamped_m = cur_m + direction * min_step
        bound = 'min'
    elif abs_delta > max_step:
        clamped_m = cur_m + direction * max_step
        bound = 'max'
    else:
        return candidate, None
    if clamped_m >= 0.99:
        return candidate, None
    new_price = wac / (1 - clamped_m)
    return round(new_price * 4) / 4, bound


ACTION_STEPS = {
    ('min', 'low'):  0,    # 1
    ('min', 'mid'):  0,    # 2
    ('min', 'high'): +2,   # 3
    ('25',  'low'):  -1,   # 4
    ('25',  'mid'):  -1,   # 5
    ('25',  'high'): +1,   # 6
    ('50',  'low'):  -2,   # 7
    ('50',  'mid'):  -1,   # 8
    ('50',  'high'): +1,   # 9
    ('75',  'low'):  -3,   # 10
    ('75',  'mid'):  -3,   # 11
    ('75',  'high'): 0,    # 12
    ('max', 'low'):  -3,   # 13
    ('max', 'mid'):  -2,   # 14
    ('max', 'high'): 0,    # 15
}


def compute_action(row):
    pos  = row['position']
    ach  = row['ach_bucket']
    cur  = row['current_price']
    cmin = row.get('commercial_min_price', 0) or 0
    wac  = row.get('wac_p', 0) or 0
    tm   = row.get('target_margin', 0.05) or 0.05

    if pos == 'no_price':
        return None, 'no_action_no_price', None
    if pos == 'below_min':
        return round(cmin * 4) / 4, 'lift_to_commercial_min', None

    tiers = build_effective_tiers(row)

    if pos == 'Target':
        cur_margin = (cur - wac) / cur if cur > 0 and wac > 0 else 0
        margin_below = cur_margin < tm
        if margin_below:
            n = +2 if ach == 'high' else -2
            base_label = f'target_marg_below_{ach}_{n:+d}'
        else:
            n = {'low': -2, 'mid': -1, 'high': 0}[ach]
            base_label = (f'target_marg_ok_{ach}_{n:+d}'
                          if n != 0 else f'target_marg_ok_{ach}_hold')
    else:
        n = ACTION_STEPS.get((pos, ach), None)
        base_label = (f'pos_{pos}_{ach}_{n:+d}'
                      if n not in (None, 0) else f'pos_{pos}_{ach}_hold')

    if n in (None, 0):
        return None, base_label, n

    candidate, clamped, no_tier = step_in_tiers(tiers, cur, n)
    if no_tier:
        return None, f'{base_label}_no_tier_{"above" if n > 0 else "below"}', n

    label = base_label + ('_clamped' if clamped else '')

    bounded, bound_hit = apply_margin_step_bounds(cur, candidate, wac, tm)
    if bound_hit:
        label += f'_step_{bound_hit}_bound'
        candidate = bounded

    return candidate, label, n


results = lookup.apply(compute_action, axis=1, result_type='expand')
results.columns = ['new_price', 'reason', 'steps']
lookup = pd.concat(
    [lookup.drop(columns=['new_price', 'reason', 'steps'], errors='ignore'), results],
    axis=1,
)

# Safety net 1: never go below 0.9 * wac_p
mask = lookup['new_price'].notna() & (lookup['wac_p'] > 0)
wac_floor = 0.9 * lookup['wac_p']
below_wac = mask & (lookup['new_price'] < wac_floor)
lookup.loc[below_wac, 'new_price'] = (wac_floor[below_wac] * 4).round() / 4

# Safety net 2: commercial_min takes precedence when present
mask  = lookup['new_price'].notna() & (lookup['commercial_min_price'].fillna(0) > 0)
below = mask & (lookup['new_price'] < lookup['commercial_min_price'])
lookup.loc[below, 'new_price'] = lookup.loc[below, 'commercial_min_price'].round(2)

print('Reason distribution (top 25):')
print(lookup['reason'].value_counts().head(25))

In [ ]:
# =============================================================================
# CELL 7b - CLASSIFICATION (informational columns only)
# This cell only adds the sales_or_margin + reason_v2 informational labels for
# review purposes. The actual price decisions are produced by the optimizer in
# Cell 7c. The matrix output from Cell 7 (new_price, reason, steps) is left
# untouched.
# =============================================================================

def classify_row(row):
    if pd.isna(row.get('nmv_30d')) or row['nmv_30d'] <= 0:
        return 'none', 'none_no_30d_sales'
    if (bool(row.get('is_bottom_quartile'))
            and row['position'] in ('min', '25')
            and float(row.get('qty_ratio', 0) or 0) > 1.5):
        return 'margin', 'margin_eligible'
    if bool(row.get('is_top50_cum')):
        return 'sales', 'sales_eligible'
    return 'none', 'none_not_in_pool'

_class = lookup.apply(
    lambda r: pd.Series(classify_row(r), index=['sales_or_margin', 'reason_v2']),
    axis=1,
)
lookup[['sales_or_margin', 'reason_v2']] = _class
lookup['new_price_v2']    = np.nan
lookup['delta_nmv_v2']    = np.nan
lookup['delta_profit_v2'] = np.nan

print('sales_or_margin distribution (informational):')
print(lookup['sales_or_margin'].value_counts(dropna=False))

In [ ]:
# =============================================================================
# CELL 7c - PROFIT-MAXIMIZING OPTIMIZER (margin-aware, demand-law respecting)
#
# Core principle: LIFTING PRICE NEVER INCREASES NMV. For lifts we use the
# historical NMV at the candidate position (with cat_brand fallback scaled by
# sku_anchor) and CAP the predicted change at zero (so noisy history can't
# project a phantom gain). Drops can grow NMV via real elasticity from history.
#
# Step range: any move walks at most STEP_CAP (=7) tiers in effective_tiers.
#
# Sales SKUs (top-50% cumulative NMV in their cohort):
#   - Find "sales-max position" from THIS SKU's own history
#     (sbp_sku only, no cat_brand fallback for the target).
#   - If current_position > sales_max_position (currently more expensive than
#     the historical sweet spot) -> drop toward sales_max_position (within -7
#     step cap, floor = max(0.9*wac, commercial_min)).
#   - Else -> hold (no lift; lifting kills sales).
#   - Never lift a sales SKU. Period.
#
# Non-sales SKUs (margin pool + 'none' pool):
#   - Lift only when qty_ratio > 1 (over-achievers; the rest may not absorb a
#     price up). For each candidate +1..+STEP_CAP within ceiling=market_max,
#     compute delta_nmv = min(0, hist_nmv_at_new - hist_nmv_at_cur) and
#     delta_profit. Pick the candidate with highest profit-per-EGP-of-NMV-cost
#     efficiency = delta_profit / max(-delta_nmv, eps). This MINIMIZES the
#     NMV drop while still capturing margin gain.
#   - Drops are used to close the remaining gap after Phase A + Phase B.
#     Use historical sales-by-position (with scale-aware fallback). Sort
#     candidates by efficiency = delta_nmv / -delta_profit; greedy.
# =============================================================================

POSITION_RANK = {
    'below_min': -1, 'min': 0, '25': 1, '50': 2, '75': 3, 'max': 4, 'Target': 2,
}

# Max steps in either direction within effective_tiers per move.
STEP_CAP = 7

def position_of_price(row, candidate):
    """Bucket a hypothetical price into our position labels using the row's
    market percentiles. Returns 'Target' if the SKU has no market data."""
    if not isinstance(row.get('price_tiers'), list) or len(row['price_tiers']) == 0 \
            or pd.isna(row.get('market_min')):
        return 'Target'
    if candidate >= row['market_max']: return 'max'
    if candidate >= row['market_75']:  return '75'
    if candidate >= row['market_50']:  return '50'
    if candidate >= row['market_25']:  return '25'
    return 'min'

def find_sku_sales_max_position(cohort, product):
    """Position label where THIS SKU sold the most (own history only,
    no cat_brand fallback). Returns None if no own history."""
    own_positions = {pos: nmv for (c, p, pos), nmv in sbp_sku.items()
                     if c == cohort and p == product and pd.notna(nmv)}
    if not own_positions:
        return None
    return max(own_positions.items(), key=lambda kv: kv[1])[0]

# ----- Compute gap from last7 baseline -----
baseline_total = float(lookup['last7_avg_daily_nmv'].sum())
gap = NMV_TARGET - baseline_total   # may be negative (already above target)
print(f'Baseline (sum of last7_avg_daily_nmv): {baseline_total:,.0f}')
print(f'Target:                                 {NMV_TARGET:,.0f}')
print(f'Initial gap:                            {gap:,.0f}')

decided = set()
running_dnmv    = 0.0
running_dprofit = 0.0

def _apply(idx, cand_price, dnmv, dprofit, reason):
    lookup.at[idx, 'new_price_v2']    = cand_price
    lookup.at[idx, 'delta_nmv_v2']    = dnmv
    lookup.at[idx, 'delta_profit_v2'] = dprofit
    lookup.at[idx, 'reason_v2']       = reason
    decided.add(idx)

# ----- Phase A - SALES SKUS: drop to historical sales-max position -----
sales_rows = lookup[lookup['sales_or_margin'] == 'sales']
phase_a_drops = 0
phase_a_holds_below_max = 0
phase_a_holds_no_history = 0

for idx, row in sales_rows.iterrows():
    cur = row.get('current_price')
    wac = float(row.get('wac_p', 0) or 0)
    if pd.isna(cur) or wac <= 0:
        continue
    cur_pos = row['position']
    if cur_pos == 'below_min':
        # The matrix already lifts these to commercial_min in Cell 7; don't
        # interfere from v2.
        lookup.at[idx, 'reason_v2'] = 'sales_below_min_handled_by_matrix'
        continue
    cmin = float(row.get('commercial_min_price', 0) or 0)
    floor = max(0.9 * wac, cmin)
    tiers = build_effective_tiers(row)
    if not tiers:
        lookup.at[idx, 'reason_v2'] = 'sales_no_tiers'
        continue

    sales_max_pos = find_sku_sales_max_position(row['cohort_id'], row['product_id'])
    if sales_max_pos is None:
        lookup.at[idx, 'reason_v2'] = 'sales_hold_no_own_history'
        phase_a_holds_no_history += 1
        continue

    cur_rank   = POSITION_RANK.get(cur_pos, 2)
    smax_rank  = POSITION_RANK.get(sales_max_pos, 2)
    if cur_rank <= smax_rank:
        # Already at or below sales-max sweet spot - lifting is forbidden
        lookup.at[idx, 'reason_v2'] = (
            f'sales_hold_at_or_below_max_(cur={cur_pos},smax={sales_max_pos})'
        )
        phase_a_holds_below_max += 1
        continue

    # Need to drop. Find the candidate (-1, -2, -3) whose new_position lands
    # closest to sales_max_pos from above (smallest |step| preferred).
    sku_anchor = float(row.get('sku_nmv_anchor', 0) or 0)
    exp_nmv_cur = expected_daily_nmv_at(
        row['cohort_id'], row['product_id'], row.get('cat'), row.get('brand'),
        cur_pos, sku_anchor,
    )
    if exp_nmv_cur is None:
        exp_nmv_cur = sku_anchor
    cur_margin = (cur - wac) / cur if cur > 0 else 0

    best = None  # (rank_distance, abs_step, cand_price, new_pos, dnmv, dprofit)
    for n in range(-1, -STEP_CAP - 1, -1):
        cand, _, _ = step_in_tiers(tiers, cur, n)
        if cand is None:
            continue
        if cand < floor:
            continue
        new_pos = position_of_price(row, cand)
        new_rank = POSITION_RANK.get(new_pos, 2)
        # we want to land at smax_rank, prefer landing exactly on it; allow
        # going below smax if no exact match (within step cap)
        rank_distance = abs(new_rank - smax_rank)
        exp_nmv_new = expected_daily_nmv_at(
            row['cohort_id'], row['product_id'], row.get('cat'), row.get('brand'),
            new_pos, sku_anchor,
        )
        if exp_nmv_new is None:
            exp_nmv_new = exp_nmv_cur   # conservative: assume no change if unknown
        new_margin = (cand - wac) / cand if cand > 0 else 0
        dnmv    = exp_nmv_new - exp_nmv_cur
        dprofit = exp_nmv_new * new_margin - exp_nmv_cur * cur_margin
        score = (rank_distance, abs(n))   # prefer exact rank match, then smallest step
        if best is None or score < best[0]:
            best = (score, cand, new_pos, dnmv, dprofit, n)

    if best is None:
        lookup.at[idx, 'reason_v2'] = 'sales_no_drop_within_floor'
        continue

    _, cand_price, new_pos, dnmv, dprofit, step = best
    _apply(idx, cand_price, dnmv, dprofit,
           f'sales_drop_to_{new_pos}_step_{int(step):+d}_smax={sales_max_pos}')
    running_dnmv    += dnmv
    running_dprofit += dprofit
    phase_a_drops += 1

print(f'Phase A (sales drops to sales-max): {phase_a_drops} drops, '
      f'{phase_a_holds_below_max} holds at-or-below-max, '
      f'{phase_a_holds_no_history} holds (no own history)')

# ----- Phase B - MARGIN LIFTS on over-achievers (qty_ratio > 1) -----
# Demand law: a lift can NEVER grow NMV. We use the historical sales-by-position
# (with cat_brand fallback scaled by sku_anchor) to estimate the expected NMV
# DROP at the candidate position, capped at zero (so noisy history can't
# project a phantom NMV gain from a lift).
#
# delta_nmv_lift    = min(0, exp_nmv_at_new_pos - exp_nmv_at_cur_pos)
# delta_profit_lift = exp_nmv_new * new_margin - exp_nmv_cur * cur_margin
#
# Among profitable lift candidates, we MINIMIZE THE DROP by picking the one
# with the highest profit-per-EGP-of-NMV-cost efficiency:
#     efficiency = delta_profit / max(-delta_nmv, eps)
# When delta_nmv == 0 (no drop), efficiency is infinite -> always preferred.
non_sales_overach = lookup[
    (lookup['sales_or_margin'] != 'sales')
    & (lookup['qty_ratio'].fillna(0) > 1)
    & (~lookup.index.isin(decided))
]
phase_b_lifts = 0
phase_b_total_drop = 0.0

for idx, row in non_sales_overach.iterrows():
    cur = row.get('current_price')
    wac = float(row.get('wac_p', 0) or 0)
    if pd.isna(cur) or wac <= 0 or cur <= wac:
        continue
    cur_pos = row['position']
    if cur_pos == 'below_min':
        continue   # let matrix handle
    has_market = isinstance(row.get('price_tiers'), list) and len(row['price_tiers']) > 0 \
                 and pd.notna(row.get('market_max'))
    ceiling = float(row['market_max']) if has_market else float('inf')

    tiers = build_effective_tiers(row)
    if not tiers:
        continue

    sku_anchor = float(row.get('sku_nmv_anchor', 0) or 0)
    exp_nmv_cur = expected_daily_nmv_at(
        row['cohort_id'], row['product_id'], row.get('cat'), row.get('brand'),
        cur_pos, sku_anchor,
    )
    if exp_nmv_cur is None or exp_nmv_cur <= 0:
        continue
    cur_margin = (cur - wac) / cur

    best = None  # (eff, dprofit, dnmv, cand_price, new_pos, step)
    for n in range(1, STEP_CAP + 1):
        cand, _, _ = step_in_tiers(tiers, cur, n)
        if cand is None or cand > ceiling:
            continue
        new_pos = position_of_price(row, cand)
        # Historical NMV at higher position (capped at zero net change per demand law)
        exp_nmv_new_raw = expected_daily_nmv_at(
            row['cohort_id'], row['product_id'], row.get('cat'), row.get('brand'),
            new_pos, sku_anchor,
        )
        if exp_nmv_new_raw is None:
            # No history at the candidate position - assume worst case is no change
            exp_nmv_new = exp_nmv_cur
        else:
            # Demand-law cap: lift cannot increase NMV
            exp_nmv_new = min(float(exp_nmv_new_raw), exp_nmv_cur)
        new_margin = (cand - wac) / cand
        dnmv    = exp_nmv_new - exp_nmv_cur            # <= 0
        dprofit = exp_nmv_new * new_margin - exp_nmv_cur * cur_margin
        if dprofit <= 0:
            continue
        cost = -dnmv                                    # >= 0
        eff  = dprofit / cost if cost > 0 else float('inf')
        if best is None or eff > best[0]:
            best = (eff, dprofit, dnmv, cand, new_pos, n)

    if best is None:
        continue
    eff, dprofit, dnmv, cand_price, new_pos, step = best
    _apply(idx, cand_price, dnmv, dprofit,
           f'margin_lift_step_{int(step):+d}_to_{new_pos}_overachiever_drop_{-dnmv:.0f}')
    running_dnmv       += dnmv          # negative or zero
    running_dprofit    += dprofit
    phase_b_total_drop += -dnmv         # positive (sum of NMV sacrificed)
    phase_b_lifts      += 1

print(f'Phase B (margin lifts on over-achievers): {phase_b_lifts} lifts, '
      f'total NMV drop {phase_b_total_drop:,.0f}')

gap_after_ab = NMV_TARGET - (baseline_total + running_dnmv)

# ----- Phase C - NON-SALES DROPS to close remaining gap -----
phase_c_drops = 0
if gap_after_ab > 0:
    drop_candidates = []
    for idx, row in lookup.iterrows():
        if idx in decided:
            continue
        if row.get('sales_or_margin') == 'sales':
            continue   # sales pool already handled in Phase A
        cur = row.get('current_price')
        wac = float(row.get('wac_p', 0) or 0)
        if pd.isna(cur) or wac <= 0:
            continue
        cur_pos = row['position']
        if cur_pos == 'below_min':
            continue
        cmin  = float(row.get('commercial_min_price', 0) or 0)
        floor = max(0.9 * wac, cmin)
        tiers = build_effective_tiers(row)
        if not tiers:
            continue
        sku_anchor = float(row.get('sku_nmv_anchor', 0) or 0)
        exp_nmv_cur = expected_daily_nmv_at(
            row['cohort_id'], row['product_id'], row.get('cat'), row.get('brand'),
            cur_pos, sku_anchor,
        )
        if exp_nmv_cur is None:
            exp_nmv_cur = sku_anchor
        cur_margin = (cur - wac) / cur if cur > 0 else 0

        best = None
        for n in range(-1, -STEP_CAP - 1, -1):
            cand, _, _ = step_in_tiers(tiers, cur, n)
            if cand is None or cand < floor:
                continue
            new_pos = position_of_price(row, cand)
            exp_nmv_new = expected_daily_nmv_at(
                row['cohort_id'], row['product_id'], row.get('cat'), row.get('brand'),
                new_pos, sku_anchor,
            )
            if exp_nmv_new is None:
                continue
            dnmv = exp_nmv_new - exp_nmv_cur
            if dnmv <= 0:
                continue
            new_margin = (cand - wac) / cand if cand > 0 else 0
            dprofit = exp_nmv_new * new_margin - exp_nmv_cur * cur_margin
            eff = dnmv / max(-dprofit, 1e-6) if dprofit < 0 else float('inf')
            if best is None or eff > best[0]:
                best = (eff, cand, new_pos, dnmv, dprofit, n)

        if best is not None:
            eff, cand, new_pos, dnmv, dprofit, n = best
            drop_candidates.append({
                'idx': idx, 'eff': eff, 'cand_price': cand, 'new_pos': new_pos,
                'dnmv': dnmv, 'dprofit': dprofit, 'step': n,
            })

    drop_candidates.sort(key=lambda d: d['eff'], reverse=True)
    remaining = gap_after_ab
    for c in drop_candidates:
        if remaining <= 0:
            break
        _apply(c['idx'], c['cand_price'], c['dnmv'], c['dprofit'],
               f'p3_close_gap_step_{int(c["step"]):+d}_to_{c["new_pos"]}')
        running_dnmv    += c['dnmv']
        running_dprofit += c['dprofit']
        remaining       -= c['dnmv']
        phase_c_drops += 1
    print(f'Phase C (non-sales drops to close gap): {phase_c_drops} drops')
else:
    print('Phase C skipped: gap already <= 0 after Phase A + B')

# Tag remaining undecided rows with usable history
mask_no_move = (~lookup.index.isin(decided)) & (lookup['nmv_30d'] > 0)
lookup.loc[mask_no_move & lookup['reason_v2'].isin(
    ['margin_eligible', 'sales_eligible', 'none_not_in_pool']
), 'reason_v2'] = 'v2_held_no_better_move'

# Floors safety net (defense-in-depth)
mask = lookup['new_price_v2'].notna() & (lookup['wac_p'] > 0)
floor_series = np.maximum(0.9 * lookup['wac_p'].fillna(0),
                          lookup['commercial_min_price'].fillna(0))
below = mask & (lookup['new_price_v2'] < floor_series)
if below.any():
    lookup.loc[below, 'new_price_v2'] = floor_series[below].round(2)
    lookup.loc[below, 'reason_v2'] = lookup.loc[below, 'reason_v2'].astype(str) + '_floored'

print()
print(f'Final delta_nmv:    {running_dnmv:,.0f}   '
      f'projected NMV total: {baseline_total + running_dnmv:,.0f}')
print(f'Final delta_profit: {running_dprofit:,.0f}')
print(f'NMV target:         {NMV_TARGET:,.0f}   '
      f'remaining gap: {max(NMV_TARGET - (baseline_total + running_dnmv), 0):,.0f}')
print()
print('reason_v2 distribution (top 25):')
print(lookup['reason_v2'].value_counts().head(25))

In [ ]:
# =============================================================================
# CELL 8 - REVIEW EXPORT
# =============================================================================
# --- Action flags per layer ---
# A row is "actionable" if EITHER layer set a price that differs from current.
# We compute three things explicitly so the filter is bulletproof against NaN
# edge cases (e.g. current_price NaN should not silently kill a v1 action).
_v1_has_price = lookup['new_price'].notna()
_v2_has_price = (
    lookup['new_price_v2'].notna()
    if 'new_price_v2' in lookup.columns
    else pd.Series(False, index=lookup.index)
)

_v1_changes_price = _v1_has_price & (lookup['new_price'].fillna(-1) != lookup['current_price'].fillna(-2))
_v2_changes_price = _v2_has_price & (lookup['new_price_v2'].fillna(-1) != lookup['current_price'].fillna(-2))

# Keep the row if EITHER layer produced an actionable price change.
# This explicitly preserves the case "v1 acted, v2 held" -> v1 row stays.
_keep = _v1_changes_price | _v2_changes_price

print(f'Action breakdown (row counts):')
print(f'  v1 (matrix) acted:      {_v1_changes_price.sum()}')
print(f'  v2 (optimizer) acted:   {_v2_changes_price.sum()}')
print(f'  BOTH acted:             {(_v1_changes_price & _v2_changes_price).sum()}')
print(f'  v1 only (v2 held):      {(_v1_changes_price & ~_v2_changes_price).sum()}')
print(f'  v2 only (v1 held):      {(~_v1_changes_price & _v2_changes_price).sum()}')
print(f'  Total kept:             {_keep.sum()}')

to_review = lookup[_keep].copy()
to_review['delta']     = to_review['new_price'] - to_review['current_price']
to_review['delta_pct'] = to_review['delta'] / to_review['current_price']
if 'new_price_v2' in to_review.columns:
    to_review['delta_v2']     = to_review['new_price_v2'] - to_review['current_price']
    to_review['delta_pct_v2'] = to_review['delta_v2'] / to_review['current_price']

# Final recommendation: prefer v2 when it acted; fall back to v1 when v2 held.
# A row reaches this point iff at least one layer has an action, so final_*
# will always be populated for every exported row.
to_review['final_price']  = to_review['new_price_v2'].where(
    to_review['new_price_v2'].notna(),
    to_review['new_price'],
)
to_review['final_reason'] = to_review['reason_v2'].where(
    to_review['new_price_v2'].notna(),
    to_review['reason'],
)
to_review['final_source'] = np.where(
    to_review['new_price_v2'].notna(), 'v2',
    np.where(to_review['new_price'].notna(), 'v1', 'hold'),
)
to_review['final_delta']     = to_review['final_price'] - to_review['current_price']
to_review['final_delta_pct'] = to_review['final_delta'] / to_review['current_price']

cols = [
    'cohort_id', 'product_id', 'sku', 'brand', 'cat',
    'wac_p', 'target_margin', 'commercial_min_price',
    'price_tiers',
    'market_min', 'market_25', 'market_50', 'market_75', 'market_max', 'market_avg',
    'current_price', 'qty_ratio', 'position', 'ach_bucket', 'steps',
    'opening_stock', 'closing_stock', 'total_stocks',
    # v1 (matrix)
    'reason', 'new_price', 'delta', 'delta_pct',
    # v2 sales/margin classification + optimizer
    'sales_or_margin', 'nmv_30d', 'nmv_share', 'last7_avg_daily_nmv',
    'is_top50_cum', 'is_bottom_quartile',
    'new_price_v2', 'delta_v2', 'delta_pct_v2',
    'delta_nmv_v2', 'delta_profit_v2', 'reason_v2',
    # Final recommendation: v2 when it acted, else v1 fallback
    'final_source', 'final_price', 'final_delta', 'final_delta_pct', 'final_reason',
]
to_review = to_review[[c for c in cols if c in to_review.columns]]

print(f'Total SKUs with action: {len(to_review)}')
print('\nfinal_source distribution (v2 wins; v1 used when v2 held):')
print(to_review['final_source'].value_counts())
print('\nReason distribution (v1 matrix):')
print(to_review['reason'].value_counts().head(15))
print('\nPosition x Ach distribution:')
print(pd.crosstab(to_review['position'], to_review['ach_bucket'], margins=True))

stamp = datetime.now(CAIRO_TZ).strftime('%Y%m%d_%H%M')
out_path = f'market_position_pricing_review_{stamp}.xlsx'
to_review.to_excel(out_path, index=False, engine='xlsxwriter')
print(f'\nReview file written: {out_path}')